# Using Pexels API for Image Collection with fastai

This notebook demonstrates how to collect and organize image datasets using the Pexels API and fastai's utilities.

## Key Concepts:
1. **Image Collection**: We use Pexels as an alternative to Bing Image Search to gather images programmatically.
   
2. **fastai's download_images**: This is a powerful utility function that:
   - Takes a list of URLs
   - Downloads them efficiently
   - Handles errors gracefully
   - Saves images directly to specified folders

3. **Directory Organization**: We create a structured dataset by:
   - Making a main directory for our project
   - Creating subdirectories for each category
   - Saving images in their respective categories

4. **Data Verification**: We use fastai's verify_images to:
   - Check for corrupted downloads
   - Remove problematic files
   - Ensure our dataset is clean and ready for model training

## Process Flow:
1. Get image URLs from Pexels API
2. Create organized folder structure
3. Download images to appropriate folders
4. Verify and clean the dataset
5. Generate a summary report

This approach sets us up for the next steps in the deep learning process by creating a well-organized image dataset that's ready for training.

Note: While the original course used Bing's API, this notebook shows how to adapt the same principles using Pexels, demonstrating how fastai's utilities work independently of the image source.

# Image Collection

Cell 1 - Setup and Imports

In [2]:
# Setup for Google Colab
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

# Import necessary libraries
import os
import requests
from fastbook import *
from fastai.vision.widgets import *
from PIL import Image

Cell 2 - Function Definition and API Key

In [3]:
def search_images_pexels(api_key, query, per_page=80):
    """Search for images on Pexels and return an L object with contentUrl attribute."""
    headers = {'Authorization': api_key}
    url = "https://api.pexels.com/v1/search"

    params = {
        'query': query,
        'per_page': per_page
    }

    response = requests.get(url, headers=headers, params=params)
    if response.status_code != 200:
        raise Exception(f"Error fetching data: {response.status_code}")

    data = response.json()
    # Create an L object to match Bing's structure
    urls = L([{'contentUrl': photo['src']['original']} for photo in data['photos']])
    return urls

# Set your Pexels API key
key = os.environ.get('PEXELS_API_KEY', 'MgHvaLKOOwPoAfmIqoC1uIhrhW0T1IsFypI2WuTOrc2SlOdf4bvS5zVv')  # Replace XXX with your key

Cell 3 - Download Images

In [ ]:
# Define your categories
bear_types = 'grizzly','black','teddy'

# Create main directory
path = Path('bears')
if not path.exists():
    path.mkdir()
    for o in bear_types:
        dest = (path/o)
        dest.mkdir(exist_ok=True)
        results = search_images_pexels(key, f'{o} bear')
        download_images(dest, urls=results.attrgot('contentUrl'))

# Verify the files were downloaded
fns = get_image_files(path)
fns

Cell 4 - Verify and Clean Images

In [5]:
# Check for corrupted images
failed = verify_images(fns)
failed

# Remove corrupted images
failed.map(Path.unlink);

Cell 5 - Summary Report

In [ ]:
# Get counts for each category
print("=== Download Summary ===")
for bear_type in bear_types:
    category_path = path/bear_type
    category_files = get_image_files(category_path)
    print(f"\n{bear_type.title()} Bears:")
    print(f"- Successfully downloaded: {len(category_files)} images")
    print(f"- Saved in: {category_path}")

print("\n=== Error Report ===")
if len(failed) > 0:
    print(f"Found and removed {len(failed)} corrupted images:")
    for f in failed:
        print(f"- {f.name}")
else:
    print("No corrupted images were found!")

print("\n=== Final Results ===")
final_files = get_image_files(path)
print(f"Total images across all categories: {len(final_files)}")
print(f"Average images per category: {len(final_files)/len(bear_types):.1f}")
print(f"\nImages are ready for use in: {path}")

# DataLoaders

Now that we have downloaded some data, we need to assemble it in a format suitable for model training. In fastai, that means creating an object called `DataLoaders`.

fastai has an extremely flexible system called the data block API. With this API you can fully customize every stage of the creation of your DataLoaders. Here is what we need to create a DataLoaders for the dataset that we just downloaded:

In [7]:
bears = DataBlock(
    blocks=(ImageBlock, CategoryBlock),     # Specifies input (images) and output (categories) data types
    get_items=get_image_files,              # Function to load all image files from the specified directory
    splitter=RandomSplitter(valid_pct=0.2, seed=42),  # Splits data into 80% training, 20% validation with fixed randomization
    get_y=parent_label,                     # Uses parent folder name as the category label for each image
    item_tfms=Resize(128)                   # Resizes all images to 128x128 pixels before processing
)

This command has given us a DataBlock object. This is like a template for creating a DataLoaders. We still need to tell fastai the actual source of our data—in this case, the path where the images can be found:

In [8]:
dls = bears.dataloaders(path)

 When you loop through a DataLoader fastai will give you 64 (by default) items at a time, all stacked up into a single tensor. We can take a look at a few of those items by calling the show_batch method on a DataLoader:

In [ ]:
dls.valid.show_batch(max_n=4, nrows=1)

By default Resize crops the images to fit a square shape of the size requested, using the full width or height. This can result in losing some important details.

In [ ]:
bears = bears.new(item_tfms=Resize(128, ResizeMethod.Squish))
dls = bears.dataloaders(path)
dls.valid.show_batch(max_n=4, nrows=1)

Alternatively, you can ask fastai to pad the images with zeros (black), or squish/stretch them:

In [ ]:
bears = bears.new(item_tfms=Resize(128, ResizeMethod.Pad, pad_mode='zeros'))
dls = bears.dataloaders(path)
dls.valid.show_batch(max_n=4, nrows=1)

Here's another example where we replace Resize with RandomResizedCrop, which is the transform that provides the behavior we just described. The most important parameter to pass in is min_scale, which determines how much of the image to select at minimum each time:

In [ ]:
bears = bears.new(item_tfms=RandomResizedCrop(128, min_scale=0.3))
dls = bears.dataloaders(path)
dls.train.show_batch(max_n=4, nrows=1, unique=True)

Examples of common data augmentation techniques for images are rotation, flipping, perspective warping, brightness changes and contrast changes.

For natural photo images such as the ones we are using here, a standard set of augmentations that we have found work pretty well are provided with the `aug_transforms` function. Because our images are now all the same size, we can apply these augmentations to an entire batch of them using the GPU, which will save a lot of time.

To tell fastai we want to use these transforms on a batch, we use the `batch_tfms` parameter (note that we're not using `RandomResizedCrop` in this example, so you can see the differences more clearly; we're also using double the amount of augmentation compared to the default, for the same reason):

In [ ]:
bears = bears.new(item_tfms=Resize(128), batch_tfms=aug_transforms(mult=2))
dls = bears.dataloaders(path)
dls.train.show_batch(max_n=8, nrows=2, unique=True)

Now that we have assembled our data in a format fit for model training, let's actually train an image classifier using it...

# Train the model

Time to use the same lines of code as in **Lecture 1** to train our bear classifier.

We don't have a lot of data for our problem (80 pictures of each sort of bear at most), so to train our model, we'll use `RandomResizedCrop` with an image size of 224 px, which is fairly standard for image classification, and default `aug_transforms`:

In [14]:
bears = bears.new(
    item_tfms=RandomResizedCrop(224, min_scale=0.5),
    batch_tfms=aug_transforms())
dls = bears.dataloaders(path)

We can now create our Learner and fine-tune it in the usual way:

In [ ]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(4)

Now let's see whether the mistakes the model is making are mainly thinking that grizzlies are teddies (that would be bad for safety!), or that grizzlies are black bears, or something else. To visualize this, we can create a confusion matrix:

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix()

> The rows represent all the black, grizzly, and teddy bears in our dataset, respectively. The columns represent the images which the model predicted as black, grizzly, and teddy bears, respectively. Therefore, the diagonal of the matrix shows the images which were classified correctly, and the off-diagonal cells represent those which were classified incorrectly. This is one of the many ways that fastai allows you to view the results of your model. It is (of course!) calculated using the validation set. With the color-coding, the goal is to have white everywhere except the diagonal, where we want dark blue. Our bear classifier isn't making many mistakes!

It's helpful to see where exactly our errors are occurring, to see whether they're due to a dataset problem (e.g., images that aren't bears at all, or are labeled incorrectly, etc.), or a model problem (perhaps it isn't handling images taken with unusual lighting, or from a different angle, etc.). To do this, we can sort our images by their loss. For now, `plot_top_losses` shows us the images with the highest loss in our dataset. As the title of the output says, each image is labeled with four things: prediction, actual (target label), loss, and probability. The probability here is the confidence level, from zero to one, that the model has assigned to its prediction:

In [ ]:
interp.plot_top_losses(5, nrows=1)

fastai includes a handy GUI for data cleaning called ImageClassifierCleaner that allows you to choose a category and the training versus validation set and view the highest-loss images (in order), along with menus to allow images to be selected for removal or relabeling:

In [19]:
# Import and initialize widgets for Colab
!pip install -q ipywidgets
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
#hide_output
cleaner = ImageClassifierCleaner(learn)
cleaner

Should we should choose <Delete> in the menu under this image. ImageClassifierCleaner doesn't actually do the deleting or changing of labels for you; it just returns the indices of items to change. So, for instance, to delete (unlink) all images selected for deletion, we would run:

In [21]:
# This will delete all images you marked for deletion
for idx in cleaner.delete():
    cleaner.fns[idx].unlink()

In [ ]:
# And this will move images to their new category if you used the "change" option
for idx,cat in cleaner.change():
    shutil.move(str(cleaner.fns[idx]), path/cat)

# Turning Your Model into an Online Application

For google colab, Gradio is the recommended solution.

In [ ]:
!pip install gradio
import gradio as gr

In [25]:
# Export the model
learn.export()

In [ ]:
# Load the exported model
learn_inf = load_learner('export.pkl')

def classify_bear(image):
    try:
        # Convert the image to fastai format
        img = PILImage.create(image)

        # Get prediction
        pred, pred_idx, probs = learn_inf.predict(img)

        # Return formatted string
        return f"Prediction: {pred}; Probability: {probs[pred_idx]:.04f}"
    except Exception as e:
        print(f"Error occurred: {str(e)}")
        return f"Error: {str(e)}"

# Create and launch the interface
demo = gr.Interface(
    fn=classify_bear,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Bear Classifier",
    description="Upload an image of a bear to classify it as grizzly, black, or teddy bear."
)

demo.launch(debug=True, share=True)